# **3. Modelado Predictivo, Línea Base y Comparación de Modelos**
---
### **Tarea 2: Split Temporal, Evaluación y Modelos Base (Línea Base)**

*Objetivos de esta fase:*
1. **Partición temporal estricta:** Implementar el corte cronológico acordado en la propuesta (Enero y Febrero de 2026 para entrenamiento/validación y Marzo de 2026 reservado estrictamente para evaluación final).
2. **Módulo de métricas (`src/evaluate.py`):** Integrar el cálculo modular y consistente de MAE, RMSE y $R^2$.
3. **Baselines obligatorios:**
   - **Baseline 0:** Mediana global histórica (referencia estática simple de la propuesta).
   - **Baseline 1:** Dummy Regressor agrupado por par origen-destino (`PULocationID` - `DOLocationID`) con fallback de mediana global.
   - **Baseline 2:** Regresión Lineal regularizada (Ridge) con pipeline de preprocesamiento de características temporales y espaciales.
4. **Tabla comparativa inicial:** Establecer la línea base cuantitativa contra la cual competirán los modelos complejos en las fases subsiguientes.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

# Módulos del proyecto
sys.path.append(os.path.abspath('..'))
from src.evaluate import evaluate_predictions, metrics_to_dataframe, save_metrics_json

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

## 3.1. Carga del Dataset Procesado
Cargamos el conjunto de datos limpio y con variables agregadas (`yellow_tripdata_processed.parquet`), preparado en la Tarea 1 sin riesgo de data leakage.

In [ ]:
DATA_PATH = os.path.join('..', 'data', 'yellow_tripdata_processed.parquet')
df = pd.read_parquet(DATA_PATH)

print(f"Dimensiones del dataset: {df.shape}")
df.head()

## 3.2. Partición Temporal Estricta (Train/Val vs Test)
Dividimos los datos cronológicamente según los objetivos:
- **Train / Validation:** Enero y Febrero de 2026.
- **Test:** Marzo de 2026 (Reservado estrictamente para evaluación final).

In [ ]:
# Asegurar formato datetime en la fecha de pickup
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])

# Definición de fecha de corte
SPLIT_DATE = pd.to_datetime('2026-03-01')

train_val_mask = df['tpep_pickup_datetime'] < SPLIT_DATE
test_mask = df['tpep_pickup_datetime'] >= SPLIT_DATE

df_train_val = df[train_val_mask].copy()
df_test = df[test_mask].copy()

# Definir variable objetivo (Target) y variables explicativas (X)
TARGET = 'trip_duration'

X_train_val = df_train_val.drop(columns=[TARGET])
y_train_val = df_train_val[TARGET]

X_test = df_test.drop(columns=[TARGET])
y_test = df_test[TARGET]

print(f"Registros Train/Val (Ene-Feb 2026): {len(df_train_val):,}")
print(f"Registros Test (Marzo 2026): {len(df_test):,}")

## 3.3. Construcción y Evaluación de Modelos Base (Baselines)
Evaluamos las 3 estrategias de referencia obligatorias utilizando la función modular `evaluate_predictions` de `src/evaluate.py`.

In [ ]:
metrics_list = []

# ==========================================
# BASELINE 0: Mediana Global Histórica
# ==========================================
global_median = y_train_val.median()
y_pred_b0 = np.full_like(y_test, fill_value=global_median, dtype=float)

metrics_b0 = evaluate_predictions(y_test, y_pred_b0, model_name="Baseline 0: Mediana Global")
metrics_list.append(metrics_b0)

# ==========================================
# BASELINE 1: Dummy por Par Origen-Destino (PULocationID - DOLocationID)
# ==========================================
group_medians = (
    df_train_val.groupby(['PULocationID', 'DOLocationID'])[TARGET]
    .median()
    .reset_index()
    .rename(columns={TARGET: 'pred_b1'})
)

df_test_b1 = df_test[['PULocationID', 'DOLocationID']].merge(
    group_medians, on=['PULocationID', 'DOLocationID'], how='left'
)

y_pred_b1 = df_test_b1['pred_b1'].fillna(global_median).values

metrics_b1 = evaluate_predictions(y_test, y_pred_b1, model_name="Baseline 1: Mediana por OD")
metrics_list.append(metrics_b1)

# ==========================================
# BASELINE 2: Regresión Lineal Regularizada (Ridge)
# ==========================================
cat_cols = ['PULocationID', 'DOLocationID']
num_cols = [c for c in X_train_val.select_dtypes(include=[np.number]).columns if c not in cat_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

ridge_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=1.0))
])

ridge_pipeline.fit(X_train_val, y_train_val)
y_pred_b2 = ridge_pipeline.predict(X_test)

metrics_b2 = evaluate_predictions(y_test, y_pred_b2, model_name="Baseline 2: Regresión Ridge")
metrics_list.append(metrics_b2)

## 3.4. Comparación de Baselines y Exportación de Métricas
Construimos la tabla comparativa inicial ($MAE$, $RMSE$ y $R^2$) y guardamos los resultados en la carpeta de reportes.

In [ ]:
# Crear y visualizar la tabla de métricas
df_baselines_metrics = metrics_to_dataframe(metrics_list)
display(df_baselines_metrics)

# Exportar métricas a formato JSON
JSON_OUTPUT_PATH = os.path.join('..', 'outputs', 'reports', 'baselines_metrics.json')
save_metrics_json(metrics_list, JSON_OUTPUT_PATH)

print(f"Resultados guardados exitosamente en: {JSON_OUTPUT_PATH}")